# ABB Marine & Ports — Vessel IMO Email Classifier

## What We Are Building

An automated **multi-class text classification pipeline** that predicts the **IMO vessel number** (International Maritime Organization 7-digit identifier) directly from inbound ABB Marine & Ports service emails — enabling the downstream CRM/ticketing system to auto-populate the vessel field without manual lookup.

Each email arrives with a subject line, body text, attachment filenames, and sender addresses. The classifier maps those composite signals to the correct vessel.

---

## The Problem

ABB Marine's global service inboxes (spare parts, Azipod support, warranty, after-sales) receive thousands of emails per month that must each be associated with a specific vessel before routing and processing can begin. Currently this is done manually, which is slow and error-prone. Many emails carry only **indirect signals** — a vessel name embedded in a PDF attachment filename, a recurring order reference code, or a known sender address pattern — that a simple keyword search cannot reliably resolve.

The previous BERT/DeBERTa-v3-small prototype was further limited by a **512-token context window**, silently truncating long email threads and suppressing the latent patterns the model needed to learn.

---

## Train of Thought

### Stage 1 — Regex Pre-filter *(high-confidence, zero-latency)*

A deterministic regex captures explicit 7-digit IMO numbers (`IMO 9319466`, bare `9319466`) from the merged email text. This stage:
- Handles **~40–60 % of inbound traffic** at near-100 % precision
- Adds negligible compute cost and always runs first
- Routes ambiguous cases forward to the neural model

### Stage 2 — ModernBERT Neural Classifier *(latent-pattern recognition)*

Emails without an unambiguous explicit IMO are classified by a fine-tuned **[ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base)** transformer (released Dec 2024, `answerdotai/ModernBERT-base`):

| Property | Value |
|---|---|
| Architecture | Modernised encoder-only BERT (Flash Attention 2, RoPE, ALiBi) |
| Parameters | 149 M |
| Native context window | **8 192 tokens** (vs. 512 in DeBERTa-v3-small) |
| HuggingFace support since | `transformers 4.46` — latest stable: **v5.3.0** (March 2026) |

The 8 192-token context eliminates the truncation problem entirely. Four email fields are concatenated with structured separator tokens — `[SUBJECT]`, `[FROM]`, `[ATTACH]`, `[BODY]` — so the model can learn field-specific patterns (e.g. an attachment named `RFQ234-22b.pdf` that is a consistent signal for a particular vessel, undetectable by regex).

### Class Imbalance Handling

The IMO label distribution is heavily long-tailed. A custom `WeightedTrainer` applies **inverse-frequency `CrossEntropyLoss`** to prevent the model from collapsing to always predicting the dominant `UNKNOWN` class. Vessels with fewer than 20 labelled training examples are consolidated into `UNKNOWN`.

### Model Registration
Register model — best checkpoint is registered in the AML model registry.
AML Command Job — optionally re-runs the same training headlessly on the gpu-t4-cluster.

### Model Deployment
Deploy endpoint — model is deployed to a ManagedOnlineEndpoint with score.py wiring Stage 1 (regex) → Stage 2 (neural) at inference time.
Endpoint test — two sample emails are sent and the JSON response is printed.

### Inference
At inference time (score.py), every email will first hit _regex_prefilter(). A hit returns confidence=1.0, source="regex_prefilter" immediately. A miss goes to the neural model, which returns a softmax confidence and sets requires_human_review=True if confidence < 70 % or predicted class is UNKNOWN.

## What We Aim to Achieve

| Metric | Target |
|---|---|
| Accuracy on named IMO classes | ≥ 90 % |
| Regex pre-filter precision | ≥ 99 % |
| End-to-end inference latency | < 500 ms p95 |
| Low-confidence fallback | `requires_human_review = true` when model confidence < 70 % |


## Setup & Configuration

In [ ]:
# ── Install required packages ─────────────────────────────────────────────────
# Latest stable versions verified via GitHub releases (March 2026):
#   PyTorch 2.10.0 | transformers 5.3.0 | accelerate 1.13.0 | datasets 4.7.0
#   mlflow 3.10.1  | scikit-learn 1.7.2 (last release supporting Python 3.10)
#
# NOTE: transformers 5.x is the current stable line. The minimum required for
# ModernBERT-base support is 4.46.0. The spec below accepts either 4.x or 5.x.
#
# On Azure ML GPU Compute, PyTorch is pre-installed with the correct CUDA build.
# Running this cell in that environment will skip reinstalling torch.
%pip install \
    "torch>=2.10.0" \
    "transformers>=4.46.0" \
    "accelerate>=1.13.0" \
    "datasets>=4.7.0" \
    "scikit-learn>=1.7.0" \
    "mlflow>=3.10.1" \
    "azureml-mlflow" \
    "azure-ai-ml" \
    "azure-identity" \
    "safetensors>=0.4.0" \
    "sentencepiece>=0.1.99" \
    "protobuf>=4.25.0" \
    "numpy<2" \
    --quiet


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import re
import json
import tempfile
from collections import Counter

# ── Data / ML ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mlflow
from datasets import Dataset
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import accuracy_score, f1_score

# ── Transformers ──────────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# ── Azure ML ──────────────────────────────────────────────────────────────────
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import (
    AmlCompute,
    CodeConfiguration,
    Data,
    Environment,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
)

# ── Version check ─────────────────────────────────────────────────────────────
import transformers as _tfm, accelerate as _acc, sklearn as _skl, datasets as _ds

print(f"torch        {torch.__version__}")
print(f"transformers {_tfm.__version__}")
print(f"accelerate   {_acc.__version__}")
print(f"datasets     {_ds.__version__}")
print(f"scikit-learn {_skl.__version__}")
print(f"mlflow       {mlflow.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}"
      + (f"  |  GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))


In [ ]:
import os
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file, if present

# ── Workspace config — read from environment ──────────────────────────────────
# Three values identify the target Azure ML workspace:
#   AZUREML_SUBSCRIPTION_ID   — Azure subscription containing the workspace
#   AZUREML_RESOURCE_GROUP    — resource group containing the workspace
#   AZUREML_WORKSPACE         — workspace name
#
# On an AML compute instance these are typically set automatically by the
# runtime; otherwise export them in your shell or set them via the JupyterLab
# launcher's "Environment variables" panel before opening the notebook.
#
# Fallback (development convenience): leave the os.environ.get(...) calls
# returning None and the assertion below will raise a clear message naming
# the missing variable instead of silently pointing at someone else's tenant.
_subscription_id   = os.environ.get("AZUREML_SUBSCRIPTION_ID")
_resource_group    = os.environ.get("AZUREML_RESOURCE_GROUP")
_workspace_name    = os.environ.get("AZUREML_WORKSPACE")

_missing = [
    name for name, val in [
        ("AZUREML_SUBSCRIPTION_ID", _subscription_id),
        ("AZUREML_RESOURCE_GROUP",  _resource_group),
        ("AZUREML_WORKSPACE",       _workspace_name),
    ] if not val
]
assert not _missing, (
    f"Missing required environment variables: {_missing}. "
    "Set them in your shell or AML compute instance environment, then restart "
    "the notebook kernel. See README_AML.md → 'Customer Quickstart' for details."
)

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=_subscription_id,
    resource_group_name=_resource_group,
    workspace_name=_workspace_name,
)
print(
    f"✓ MLClient bound to workspace '{_workspace_name}' "
    f"(rg: {_resource_group})"
)


AssertionError: Missing required environment variables: ['AZUREML_SUBSCRIPTION_ID', 'AZUREML_RESOURCE_GROUP', 'AZUREML_WORKSPACE']. Set them in your shell or AML compute instance environment, then restart the notebook kernel. See README_AML.md → 'Customer Quickstart' for details.

## Load & Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from datasets import Dataset
from sklearn.model_selection import train_test_split as sk_split

dataset_path = "dataset_onlyWithImoUntil2023.csv"  # Training set (pre-2023 holdout)
csv_sep = ";"

UNKNOWN_LABEL = "UNKNOWN"
min_examples_per_imo = 20  # Vessels with fewer emails become UNKNOWN class

# Load CSV (only needed columns)
usecols = ["emailSubject", "emailBody", "Attachments", "emailAddresses", "IMO"]
df = pd.read_csv(dataset_path, sep=csv_sep, usecols=usecols, dtype=str, low_memory=False)

def normalize_imo(value: str) -> str:
    if value is None:
        return UNKNOWN_LABEL
    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return UNKNOWN_LABEL
    s = s.replace("IMO", "").replace("imo", "")
    if s.endswith(".0"):
        s = s[:-2]
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) < 6:
        return UNKNOWN_LABEL
    return "IMO" + digits

def merge_fields(row) -> str:
    """
    Merge email fields with structured separator tokens so the model can
    distinguish attachment filenames from body text.  This is critical for
    learning latent patterns such as  'RFQ234-22b.pdf' → specific vessel,
    a signal invisible to humans but discoverable by ML with enough data.
    """
    subject     = str(row.get("emailSubject",   "") or "").strip()
    addresses   = str(row.get("emailAddresses", "") or "").strip()
    attachments = str(row.get("Attachments",    "") or "").strip()
    body        = str(row.get("emailBody",      "") or "").strip()
    parts = []
    if subject:     parts.append(f"[SUBJECT] {subject}")
    if addresses:   parts.append(f"[FROM] {addresses}")
    if attachments: parts.append(f"[ATTACH] {attachments}")
    if body:        parts.append(f"[BODY] {body}")
    return " ".join(parts)

df["label_imo"] = df["IMO"].map(normalize_imo)
df["text"]      = df.apply(merge_fields, axis=1)

df = df[["text", "label_imo"]].dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]

dataset = Dataset.from_pandas(df, preserve_index=False)

# Closed-set: only IMOs with enough examples become named classes; rest → UNKNOWN
imo_counts    = Counter(dataset["label_imo"])
frequent_imos = sorted([
    imo for imo, c in imo_counts.items()
    if imo != UNKNOWN_LABEL and c >= min_examples_per_imo
])

label_list  = frequent_imos + [UNKNOWN_LABEL]
label_to_id = {label: idx for idx, label in enumerate(label_list)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_labels  = len(label_list)

def encode_label(example):
    imo = example["label_imo"]
    if imo == UNKNOWN_LABEL or imo_counts.get(imo, 0) < min_examples_per_imo:
        imo = UNKNOWN_LABEL
    return {"label": label_to_id[imo]}

dataset = dataset.map(encode_label)

unknown_mapped = sum(
    1 for imo in dataset["label_imo"]
    if imo == UNKNOWN_LABEL or imo_counts.get(imo, 0) < min_examples_per_imo
)

print(f"Rows loaded:                       {len(dataset)}")
print(f"Named IMO classes (≥{min_examples_per_imo} examples): {len(frequent_imos)}")
print(f"UNKNOWN-mapped rows:               {unknown_mapped}")
print(f"num_labels:                        {num_labels}")

# Stratified split — ensures every named class appears in train AND validation
df_labeled = dataset.to_pandas()
train_idx, val_idx = sk_split(
    range(len(df_labeled)),
    test_size=0.2,
    stratify=df_labeled["label"],
    random_state=42,
)
train_ds = dataset.select(train_idx)
val_ds   = dataset.select(val_idx)
print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")


### Stage 1: Regex Pre-filter (High-Confidence IMO Extraction)

A lightweight regex pass identifies **explicit IMO numbers** in email text before any neural inference. At serving time this stage fires first and only passes ambiguous cases to the ModernBERT classifier.

Two hardening measures are applied to reach the ≥ 99 % precision target:

1. **Explicit `IMO` prefix required** — the regex now matches `IMO 9319466`, `IMO9319466`, `IMO#9319466`, etc. Bare 7-digit numbers (order refs, part codes, date codes) are no longer matched by the regex and are handled by the neural model instead.

2. **IMO check-digit validation (Resolution A.600(15))** — every candidate is validated against the mandatory IMO check digit: multiply digits 1–6 by weights 7, 6, 5, 4, 3, 2, sum the products; the units digit must equal digit 7. Order numbers and part codes almost never satisfy this rule, eliminating the remaining coincidental collisions.

A **diagnostic block** prints the first 20 mismatched emails so common error patterns (sister-vessel references, forwarded threads carrying a prior IMO, order codes that collide with a known IMO) can be identified and addressed in data cleaning if needed.

The hit rate after tightening will be lower (≈ 3–5 %) — this is expected and acceptable. The neural model is designed to carry the remaining 95–97 % of traffic, and a smaller but near-perfect regex fast path is preferable to a larger but noisy one.

In [ ]:
import re

# Require explicit "IMO" prefix — bare 7-digit numbers (order refs, part codes,
# date codes, etc.) are no longer matched and are handled by the neural model.
_IMO_RE = re.compile(r'\bIMO[\s#:\-]?(\d{7})\b', re.IGNORECASE)
_known_imo_set = set(frequent_imos)  # built from training data in previous cell


def _imo_checksum_valid(digits: str) -> bool:
    """
    Validates a 7-digit string against the IMO check-digit rule
    (IMO Resolution A.600(15)).  Weights 7,6,5,4,3,2 are applied to
    digits 1-6; the sum mod 10 must equal the 7th digit.
    Order numbers and part codes almost never pass this check.
    """
    if len(digits) != 7 or not digits.isdigit():
        return False
    weights = [7, 6, 5, 4, 3, 2]
    total = sum(int(digits[i]) * weights[i] for i in range(6))
    return (total % 10) == int(digits[6])


def extract_explicit_imo(text: str) -> str | None:
    """
    Returns a known IMO string ('IMO1234567') if exactly one unambiguous known
    IMO number is found in the email text, else None (routes to classifier).
    Guards applied in order:
      1. Explicit 'IMO' prefix required (tightened regex)
      2. IMO check-digit must be valid (eliminates coincidental 7-digit collisions)
      3. Must be in the known training-set IMO list
      4. Ambiguous matches (two different known IMOs) return None
    """
    raw_matches = _IMO_RE.findall(text)
    known_hits = {
        f"IMO{m}" for m in raw_matches
        if _imo_checksum_valid(m) and f"IMO{m}" in _known_imo_set
    }
    return known_hits.pop() if len(known_hits) == 1 else None


# ── Measure pre-filter coverage on the full dataset ───────────────────────────
df_full = dataset.to_pandas()
df_full["explicit_imo"] = df_full["text"].apply(extract_explicit_imo)

total     = len(df_full)
hit_count = df_full["explicit_imo"].notna().sum()
hit_pct   = hit_count / total * 100
print(f"Regex pre-filter hits: {hit_count:,} / {total:,}  ({hit_pct:.1f}%)")
print(f"Emails routed to ML classifier: {total - hit_count:,}  ({100 - hit_pct:.1f}%)")

# Sanity check — do pre-filter hits agree with ground-truth labels?
agree     = (df_full["explicit_imo"] == df_full["label_imo"]).sum()
agree_pct = agree / hit_count * 100 if hit_count else 0
target_met = "✓ PASS" if agree_pct >= 99.0 else "✗ FAIL — inspect mismatches below"
print(f"Pre-filter accuracy on its own hits: {agree_pct:.1f}%  (target ≥99%)  {target_met}")

# ── Diagnose mismatches ────────────────────────────────────────────────────────
# Common patterns: sister-vessel IMO in a forwarded thread, order codes that
# coincidentally collide with a known IMO, emails referencing multiple vessels.
mismatches = df_full[
    df_full["explicit_imo"].notna() &
    (df_full["explicit_imo"] != df_full["label_imo"])
].copy()
print(f"\nMismatched emails (pre-filter ≠ ground-truth label): {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches[["text", "label_imo", "explicit_imo"]].head(20).to_string())


### Tokenization

In [ ]:
from transformers import AutoTokenizer

# ModernBERT-base (Dec 2024): 8192-token native context window vs 512 for DeBERTa.
# Long email threads are silently truncated at 512 tokens — a primary cause of
# low accuracy on the existing prototype. ModernBERT fixes this entirely.
model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Use 2048 tokens as a practical balance of coverage vs. batch memory.
# The model supports up to 8192 — increase if GPU VRAM > 24 GB.
MAX_SEQ_LEN = 2048
tokenizer.model_max_length = MAX_SEQ_LEN

def tokenize_batch(batch):
    # No padding here — DataCollatorWithPadding in the Trainer pads each batch
    # to its longest sample instead of the global maximum (2048).  For ABB Marine
    # emails averaging ~400–600 tokens this saves 3–5× attention compute per step,
    # cutting per-epoch wall time roughly in half on a single T4.
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

# Keep 'text' out of remove_columns — datasets 4.x withholds those columns
# from the input batch before calling the map function (lazy-evaluation optimisation).
train_remove_cols = [c for c in train_ds.column_names if c not in ("label", "text")]
val_remove_cols   = [c for c in val_ds.column_names   if c not in ("label", "text")]

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=train_remove_cols)
val_ds   = val_ds.map(tokenize_batch,   batched=True, remove_columns=val_remove_cols)

# Drop 'text' now that tokenisation is complete — it's no longer needed
train_ds = train_ds.remove_columns(["text"])
val_ds   = val_ds.remove_columns(["text"])

print(f"Train tokenized: {train_ds}")
print(f"Val   tokenized: {val_ds}")


### Fine-tune transformer model

In [ ]:
import os
import torch
import mlflow
from torch import nn
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers.integrations import MLflowCallback
from sklearn.metrics import accuracy_score, f1_score

# ── Close any stale MLflow run from a previous (crashed) attempt ───────────────
# AzureML MLflow does not allow re-logging a parameter key in the same run.
# If this cell is re-run after an OOM or other crash, the previous run will still
# be active and the Trainer's MLflowCallback will fail on log_params.
# mlflow.end_run() is a no-op when no run is active, so it is always safe to call.
mlflow.end_run()

# ── Single-GPU enforcement ─────────────────────────────────────────────────────
# The Trainer auto-wraps in DataParallel when multiple CUDA devices are visible,
# making GPU 0 gather all outputs and OOM.  The AML DDP job uses torchrun instead.
# Force single-GPU for notebook-level training to avoid this.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── Class-balanced loss ────────────────────────────────────────────────────────
# The UNKNOWN class will dominate the label distribution (many emails with
# infrequent or missing IMOs).  Without weighting the model collapses to always
# predicting UNKNOWN.  Inverse-frequency weights fix this.
label_counts  = Counter(train_ds["label"])
total_train   = sum(label_counts.values())
class_weights = torch.tensor(
    [total_train / (num_labels * max(label_counts.get(i, 1), 1)) for i in range(num_labels)],
    dtype=torch.float,
)
print(f"Class weight range: min={class_weights.min():.3f}, max={class_weights.max():.3f}")


class WeightedTrainer(Trainer):
    """Trainer subclass that applies per-class inverse-frequency weighting."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Azure ML MLflow param-length fix ───────────────────────────────────────────
# Azure ML's MLflow backend enforces a 500-char limit on parameter values — much
# stricter than the open-source MLflow client limit of 6000 chars.  The default
# MLflowCallback serialises model.config (including id2label / label2id dicts)
# as params, which easily exceeds 500 chars with >30 IMO classes.
# MLFLOW_TRUNCATE_LONG_VALUES only truncates to 6000 — useless here.
# Fix: temporarily swap the large dicts with short placeholders during setup(),
# then restore them so checkpoints contain the correct label maps.

class AzureMLSafeCallback(MLflowCallback):
    """MLflowCallback that prevents Azure ML's 500-char param limit from
    rejecting the large id2label / label2id dicts logged from model.config."""
    def setup(self, args, state, model):
        _id2label = model.config.id2label
        _label2id = model.config.label2id
        model.config.id2label = {0: f"{len(_id2label)} classes - see checkpoint config.json"}
        model.config.label2id = {"see_config_json": 0}
        try:
            super().setup(args, state, model)
        finally:
            model.config.id2label = _id2label
            model.config.label2id = _label2id


# ── Model ──────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }


# ── Memory budget for a single T4 (14.58 GiB) at seq_len=2048 ─────────────────
# Dynamic padding (DataCollatorWithPadding) pads each batch to its longest sample
# instead of the global max (2048 tokens).  For ABB Marine emails averaging
# ~400–600 raw tokens, effective attention sequence length per batch drops to
# ~600–800 tokens, cutting attention compute by 3–5× vs. max_length padding.
# Combined with gradient_checkpointing, this fits comfortably on a single T4.
_num_epochs          = 15
_batch_size          = 2   # per-device (was 8 — OOM at seq_len=2048 on T4)
_grad_accum_steps    = 16  # effective batch = 2 × 16 = 32
_warmup_ratio        = 0.06
_total_steps         = _num_epochs * (len(train_ds) // (_batch_size * _grad_accum_steps))
_warmup_steps        = max(1, int(_warmup_ratio * _total_steps))
print(f"Effective batch size: {_batch_size * _grad_accum_steps}")
print(f"Total optimizer steps: {_total_steps}  |  Warmup steps (6%): {_warmup_steps}")

# ── Training arguments ─────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./outputs-ModernBERT",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=_num_epochs,
    per_device_train_batch_size=_batch_size,
    per_device_eval_batch_size=4,           # keep eval batch small too
    gradient_accumulation_steps=_grad_accum_steps,
    gradient_checkpointing=True,            # recompute activations — ~60% VRAM saving
    learning_rate=3e-5,
    warmup_steps=_warmup_steps,             # warmup_ratio removed in transformers v5.2
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),         # mixed precision
    dataloader_num_workers=2,
    report_to="mlflow",
    save_total_limit=3,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,             # 'tokenizer=' renamed in transformers v5
    data_collator=DataCollatorWithPadding(tokenizer),  # dynamic per-batch padding
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Swap default MLflowCallback for Azure-safe version that avoids 500-char limit
trainer.remove_callback(MLflowCallback)
trainer.add_callback(AzureMLSafeCallback())

trainer.train()
metrics = trainer.evaluate()
print(metrics)
print(f"\n✓ Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"✓ Best eval_f1:    {trainer.state.best_metric:.4f}")

### Retrieve the registered model

Registration is performed **once**, automatically, by `train.py` at the end of the AML Command Job via `mlflow.transformers.log_model(..., registered_model_name="imo-extractor-model")`. This cell only *fetches* the latest registered version so the deploy cell below has a `model_details` handle to reference — there is no second write path.

> Run this **after** the AML Command Job below has finished. If you only ran the in-notebook training cells above (quick-iteration mode), no AML-registered model exists yet and this cell will raise — that's intentional: the in-notebook path is for development, not deployment.

In [ ]:
# Single source of truth for model registration: train.py logs and registers
# the trained checkpoint via `mlflow.transformers.log_model(registered_model_name=
# "imo-extractor-model", ...)` on rank 0 at the end of the AML Command Job.
#
# This cell intentionally does NOT call `ml_client.models.create_or_update(...)` —
# doing so would create a duplicate registry version on every run and make
# "which version is in production?" ambiguous. We only *retrieve* the latest
# auto-registered version here so the deployment cell below has a `model_details`
# handle to consume.
model_details = ml_client.models.get(name="imo-extractor-model", label="latest")
print(
    f"Resolved registered model: {model_details.name} "
    f"v{model_details.version}  (latest auto-registered by train.py)"
)
if model_details.tags:
    print(f"  tags: {model_details.tags}")

---

## AML Command Job: Full-Scale GPU Training

The cells above run training **inside this notebook kernel** — useful for quick iteration on a small sample. For the full ~100 K-email dataset the recommended path is an AML Command Job: the run is captured in *Jobs* with code snapshot, input-data lineage, environment lineage, driver logs, and an auto-registered model version tied back to the producing job. None of that exists if you just run `python train.py` over SSH.

This cell picks the cheapest compute target available without a quota request:

1. **If `AML_TRAIN_COMPUTE` is set**, the job runs on that pre-existing compute (typically a Compute Instance name). Useful when you already have a GPU CI provisioned and want zero cluster-quota cost.
2. **Otherwise**, it provisions an auto-scaling **1× T4 cluster** (`Standard_NC16as_T4_v3`, 16 cores) — the largest T4 SKU that fits inside a 20-core `Standard NCASv3_T4 Family Cluster Dedicated vCPUs` quota. `min_instances=0` so you only pay while a job runs.

`train.py` is launched via `torchrun --standalone --nproc_per_node=1`, which keeps the DDP-aware codepaths (`is_main_process`, MLflow rank-0 guards) active with a single worker. The trained checkpoint is identical to the 4-GPU DDP path at the same effective batch size — just ~4× slower wall-clock.

**Single-GPU settings applied:**
- `--batch-size 8` per device → effective batch size = 8 (1 GPU × 8)
- `--lr 3e-5` (single-GPU baseline)
- All MLflow logging and model registration are guarded to rank 0 only in `train.py`

**Want 4× T4 DDP instead?** Request `Standard NCASv3_T4 Family Cluster Dedicated vCPUs` ≥ 64 in the workspace region, then switch the cluster SKU to `Standard_NC64as_T4_v3` and the command to `torchrun --nproc_per_node=4` with `--lr 6e-5` (conservative √4 scaling).

**Training / serving hardware alignment.** Training runs on T4 (Turing, sm_75) and the production endpoint below targets a single T4 of the same generation. The trained checkpoint is plain fp32 `.safetensors` and is hardware-portable — `train.py` will automatically pick `bf16` on Ampere+ (A100/H100), `fp16` on T4/V100, or `fp32` on CPU at train time, with no impact on the deployed model.

Monitor progress in **AML Studio → Experiments → imo-extractor-experiment** and in **Jobs → modernbert-imo-1gpu**.


In [ ]:
import os
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AmlCompute, Data, Environment

# ── 1. Compute target ─────────────────────────────────────────────────────────
# Resolution order (first match wins):
#   1. AML_TRAIN_COMPUTE env var — name of an existing compute (Compute Instance
#      or AmlCompute). Lets you point the job at a GPU CI you already have,
#      bypassing the cluster-dedicated quota entirely.
#   2. Auto-provisioned single-GPU cluster `gpu-t4-single` on
#      Standard_NC16as_T4_v3 (1× T4, 16 cores) — fits inside the customer's
#      20-core `Standard NCASv3_T4 Family Cluster Dedicated vCPUs` quota.
#
# Why not Standard_NC64as_T4_v3 (4× T4)? It needs 64 cluster-dedicated cores.
# If the customer has < 64 cores of that quota line, cluster creation fails
# regardless of how many nodes you ask for. See aml/README.md for the quota
# table and request instructions.
COMPUTE_TARGET = os.environ.get("AML_TRAIN_COMPUTE", "").strip()

if COMPUTE_TARGET:
    compute_name = COMPUTE_TARGET
    print(f"✓ Using pre-existing compute: {compute_name} (from AML_TRAIN_COMPUTE)")
else:
    compute_name = "gpu-t4-single"
    gpu_cluster = AmlCompute(
        name=compute_name,
        type="amlcompute",
        size="Standard_NC16as_T4_v3",     # 1× T4, 16 cores — fits 20-core quota
        min_instances=0,                  # Scale to zero when idle (no idle cost)
        max_instances=1,
        idle_time_before_scale_down=120,  # 2 min idle before shutdown
        tier="dedicated",
    )
    gpu_cluster = ml_client.begin_create_or_update(gpu_cluster).result()
    print(f"✓ Compute cluster: {gpu_cluster.name}  state={gpu_cluster.provisioning_state}")

# ── 2. Register training CSV as versioned Data Asset ──────────────────────────
# Canonical training file: dataset_onlyWithImoUntil2023.csv (114 K rows, ';' separator).
# The temporal holdout dataset_onlyWithImoFrom2023.csv is NOT registered here —
# it is the never-seen test set used by the Tier-5 evaluation script.
train_data_asset = ml_client.data.create_or_update(
    Data(
        name="email-imo-training-data",
        path="./dataset_onlyWithImoUntil2023.csv",
        type=AssetTypes.URI_FILE,
        description="ABB Marine email → IMO labelled dataset (pre-2023, ';'-separated)",
        version="1",
    )
)
print(f"✓ Data asset: {train_data_asset.name} v{train_data_asset.version}")

# ── 3. AML Environment from conda.yml ─────────────────────────────────────────
# Base image must ship CUDA 12.4+ to host the torch>=2.10 wheels pinned in
# conda.yml. The previous CUDA 11.8 base image could not satisfy the wheel's
# CUDA runtime requirement and would fall back to a CPU-only build.
train_env = ml_client.environments.create_or_update(
    Environment(
        name="imo-classifier-train-env",
        conda_file="./conda.yml",
        image="mcr.microsoft.com/azureml/openmpi5.0-cuda12.4-cudnn9-ubuntu22.04:latest",
        description="ModernBERT fine-tuning: torch 2.10 / cu124, transformers>=4.46",
    )
)
print(f"✓ Environment: {train_env.name}")

# ── 4. Submit Command Job (torchrun, single GPU) ──────────────────────────────
# `torchrun --standalone --nproc_per_node=1` keeps train.py's DDP-aware
# codepaths active (RANK/WORLD_SIZE set) with a single worker, so this run
# exercises the same control flow as a future 4-GPU DDP job — just without
# the extra ranks. HuggingFace Trainer detects the single-worker setup and
# skips the DDP wrapper automatically.
#
# Batch size & LR (single-GPU baseline):
#   --batch-size 8  →  effective batch = 1 × 8 = 8
#   --lr 3e-5       →  baseline LR (no √k scaling, since k=1)
#
# To upgrade to 4-GPU DDP later: switch compute to Standard_NC64as_T4_v3,
# change torchrun to `--nproc_per_node=4`, and bump `--lr 6e-5`.
job = command(
    inputs={
        "training_data": Input(
            type=AssetTypes.URI_FILE,
            path=f"azureml:{train_data_asset.name}:{train_data_asset.version}",
        )
    },
    outputs={
        "model_dir": Output(type=AssetTypes.URI_FOLDER)
    },
    code="./",                       # Uploads entire working directory as code snapshot
    command=(
        "torchrun --standalone --nproc_per_node=1 train.py "
        "--data ${{inputs.training_data}} "
        "--csv-sep ';' "
        "--model-name answerdotai/ModernBERT-base "
        "--output-dir ${{outputs.model_dir}} "
        "--epochs 15 "
        "--batch-size 8 "
        "--lr 3e-5 "
        "--max-seq-len 2048 "
        "--min-examples 20 "
    ),
    environment=f"azureml:{train_env.name}@latest",
    compute=compute_name,
    experiment_name="imo-extractor-experiment",
    display_name="modernbert-imo-1gpu",
    description=f"ModernBERT-base fine-tuning on 1× T4 (compute={compute_name})",
    tags={
        "model":    "answerdotai/ModernBERT-base",
        "task":     "imo-classification",
        "dataset":  train_data_asset.name,
        "training": "1-gpu",
        "compute":  compute_name,
    },
)

returned_job = ml_client.jobs.create_or_update(job)
print(f"\n✓ Job submitted: {returned_job.name}")
print(f"  Studio URL: {returned_job.studio_url}")
print(f"\nStreaming logs (Ctrl+C to detach — job continues in background):")
ml_client.jobs.stream(returned_job.name)


---

## Architecture bake-off (optional)

The single-job cell above trains **ModernBERT-base** — the current production baseline. The two cells below submit a **4-candidate bake-off** on identical splits so the customer can compare modern encoders before committing the chosen one to the deployment cells further down.

Candidates (see also the "Corpus language profile" section in `README.md`):

| Alias              | HF model ID                       | Role                |
|--------------------|-----------------------------------|---------------------|
| `modernbert-base`  | `answerdotai/ModernBERT-base`     | Baseline            |
| `modernbert-large` | `answerdotai/ModernBERT-large`    | Scale comparison    |
| `deberta-v3-base`  | `microsoft/deberta-v3-base`       | Short-ctx control   |
| `eurobert-210m`    | `EuroBERT/EuroBERT-210M`          | Multilingual probe  |

**Compute & cost:** the cell re-uses whichever target the single-job cell resolved (`COMPUTE_TARGET` from `AML_TRAIN_COMPUTE` env var, else the auto-provisioned `gpu-t4-single` cluster on `Standard_NC16as_T4_v3`). On a single-GPU cluster with `max_instances=1`, AML queues the four jobs serially — total wall-clock ≈ 4× a single run (~8–16 h end-to-end on 1× T4). To get them running truly in parallel you'd need cluster-dedicated quota for `max_instances=4` or a larger SKU.

**Behaviour:**
- Each job logs to the same MLflow experiment (`imo-extractor-experiment`) with a distinct `arch_alias` tag.
- Each job registers under its own model name (`imo-extractor-{alias}`) — leaving `imo-extractor-model` (used by the deploy cell) untouched until the customer picks a winner.
- The submitter does **not** stream logs; it fires the four jobs and prints their studio URLs. Watch progress in AML Studio.
- Re-running this cell is safe — it submits new job runs (AML auto-generates run names).


In [ ]:
# Architecture bake-off submitter — fires 4 Command Jobs (single-GPU each).
#
# Re-uses the same compute target, data asset and environment as the single-job
# cell above. Each candidate gets its own job tags, MLflow `arch_alias`, and
# registered model name so the four runs are easy to disentangle later.
#
# Per-device batch sizes are tuned for a single T4 (16 GB VRAM); they also work
# on larger SKUs unchanged. Drop --lr to 3e-5 (single-GPU baseline) — the
# previous 6e-5 was a √4 scaling for 4-GPU DDP.
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

BAKEOFF_CANDIDATES = [
    # (alias,            hf_model_id,                       per_device_bs, max_seq_len)
    ("modernbert-base",  "answerdotai/ModernBERT-base",     8,             2048),
    ("modernbert-large", "answerdotai/ModernBERT-large",    4,             2048),  # ~2.6× params → halve batch
    ("deberta-v3-base",  "microsoft/deberta-v3-base",       16,            512),   # short ctx → bigger batch
    ("eurobert-210m",    "EuroBERT/EuroBERT-210M",          8,             2048),
]

submitted_runs = []
for alias, hf_id, per_device_bs, max_seq in BAKEOFF_CANDIDATES:
    bakeoff_job = command(
        inputs={
            "training_data": Input(
                type=AssetTypes.URI_FILE,
                path=f"azureml:{train_data_asset.name}:{train_data_asset.version}",
            )
        },
        outputs={"model_dir": Output(type=AssetTypes.URI_FOLDER)},
        code="./",
        command=(
            "torchrun --standalone --nproc_per_node=1 train.py "
            "--data ${{inputs.training_data}} "
            "--csv-sep ';' "
            f"--model-name {hf_id} "
            f"--arch-alias {alias} "
            f"--registered-model-name imo-extractor-{alias} "
            "--output-dir ${{outputs.model_dir}} "
            "--epochs 15 "
            f"--batch-size {per_device_bs} "
            "--lr 3e-5 "
            f"--max-seq-len {max_seq} "
            "--min-examples 20 "
        ),
        environment=f"azureml:{train_env.name}@latest",
        compute=compute_name,
        experiment_name="imo-extractor-experiment",
        display_name=f"bakeoff-{alias}",
        description=f"Architecture bake-off — {alias} ({hf_id})",
        tags={
            "model":       hf_id,
            "arch_alias":  alias,
            "task":        "imo-classification",
            "dataset":     train_data_asset.name,
            "training":    "1-gpu",
            "compute":     compute_name,
            "bakeoff":     "v1",
        },
    )
    run = ml_client.jobs.create_or_update(bakeoff_job)
    submitted_runs.append((alias, run.name, run.studio_url))
    print(f"✓ Submitted {alias:18s}  job={run.name}")
    print(f"   {run.studio_url}")

print(f"\n{len(submitted_runs)} bake-off jobs submitted. Track progress in AML Studio.")
print("Re-run the next cell once all four have finished to see the comparison table.")


### Bake-off results aggregator

Run this cell **after** all four bake-off jobs above show `Completed` in AML Studio. It queries the MLflow tracking server attached to this workspace, pulls every run tagged `bakeoff=v1`, and produces a single comparison table sorted by validation F1.

Pick the winner by:
1. Highest `eval_f1` (primary metric — matches `load_best_model_at_end=True`)
2. `eval_accuracy` as a tie-breaker
3. Wall-clock / cost as the deciding factor when metrics are within ~1 pp

Once a winner is picked, re-submit just that candidate using the single-job cell at the top of this section but with `--registered-model-name imo-extractor-model`, so the deployment cells further down find it under their expected name.


In [ ]:
# Bake-off results aggregator — pulls the 4 MLflow runs into a comparison table.
#
# Uses the MLflow tracking URI configured by the AML workspace context (set by
# `azureml-mlflow` on a compute instance). Falls back gracefully if no runs are
# found yet (i.e. jobs still running).
import mlflow
import pandas as pd

# Point MLflow at the AML workspace tracking server (no-op if already configured).
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

experiment = mlflow.get_experiment_by_name("imo-extractor-experiment")
if experiment is None:
    raise RuntimeError(
        "Experiment 'imo-extractor-experiment' not found. "
        "Submit the bake-off jobs above first."
    )

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.bakeoff = 'v1'",
    output_format="pandas",
)

if runs.empty:
    print("No bake-off runs found yet. Wait for the four jobs above to finish, then re-run this cell.")
else:
    # Keep one row per arch_alias — the most recent FINISHED run wins if the cell
    # was submitted multiple times.
    finished = runs[runs["status"] == "FINISHED"].copy()
    if finished.empty:
        print(f"Found {len(runs)} bake-off runs but none have finished yet. "
              f"Statuses: {runs['status'].value_counts().to_dict()}")
    else:
        finished = finished.sort_values("end_time", ascending=False)
        finished = finished.drop_duplicates(subset=["tags.arch_alias"], keep="first")

        # Columns of interest. Some may be missing depending on transformers version
        # (e.g. eval_loss vs metrics.eval_loss); guard each one.
        metric_cols = [
            "metrics.final_eval_f1",
            "metrics.final_eval_accuracy",
            "metrics.eval_loss",
            "metrics.prefilter_coverage_pct",
        ]
        param_cols = [
            "params.arch_alias",
            "params.model_name",
            "params.batch_size",
            "params.lr",
            "params.max_seq_len",
            "params.num_labels",
            "params.registered_model_name",
        ]
        misc_cols = ["run_id", "start_time", "end_time"]

        available = [c for c in metric_cols + param_cols + misc_cols if c in finished.columns]
        comparison = finished[available].copy()

        # Wall-clock minutes for cost intuition
        if {"start_time", "end_time"}.issubset(comparison.columns):
            comparison["wall_clock_min"] = (
                (pd.to_datetime(comparison["end_time"]) - pd.to_datetime(comparison["start_time"]))
                .dt.total_seconds() / 60
            ).round(1)

        # Sort by primary metric descending so the winner is on top
        sort_col = "metrics.final_eval_f1" if "metrics.final_eval_f1" in comparison.columns else None
        if sort_col:
            comparison = comparison.sort_values(sort_col, ascending=False)

        # Drop noisy timestamp/run_id columns from the printed view but keep run_id last
        display_cols = [c for c in comparison.columns if c not in ("start_time", "end_time")]
        print(f"Bake-off comparison ({len(comparison)} / {len(BAKEOFF_CANDIDATES)} candidates finished):\n")
        with pd.option_context("display.max_columns", None, "display.width", 200):
            print(comparison[display_cols].to_string(index=False))

        # Persist for the customer's records
        comparison.to_csv("bakeoff_comparison.csv", index=False)
        print("\n✓ Saved to bakeoff_comparison.csv")


###  Deploy as Managed Online Endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Environment, CodeConfiguration

endpoint_name = "imo-extractor-endpoint"
endpoint = ManagedOnlineEndpoint(name=endpoint_name, auth_mode="key")
ml_client.begin_create_or_update(endpoint).wait()

# Inference environment — MUST be a CUDA-enabled base image so the deployment
# can see the T4 GPU on the Standard_NC*_T4_v3 instance below. The previous
# openmpi4.1.0-ubuntu22.04 base ships no CUDA runtime; ModernBERT would silently
# fall back to CPU and take ~5–15 s per request (10–30× the 500 ms p95 target).
# Re-using the same CUDA 12.4 base as training keeps conda.yml valid unchanged.
env = Environment(
    name="imo-classifier-inference",
    conda_file="conda.yml",
    image="mcr.microsoft.com/azureml/openmpi5.0-cuda12.4-cudnn9-ubuntu22.04:latest",
    description="ModernBERT inference: torch 2.10 / cu124, GPU runtime",
)

deployment = ManagedOnlineDeployment(
    name="default",
    endpoint_name=endpoint_name,
    model=model_details,
    environment=env,
    code_configuration=CodeConfiguration(code="./", scoring_script="score.py"),
    # ── Inference SKU — Standard_NC16as_T4_v3 (1× T4, 16 GB VRAM) ────────────
    # Single-request inference does not benefit from multi-GPU; a single T4 is
    # ample for ModernBERT-base (~600 MB weights + ~1.5 GB activations at
    # seq_len=2048 fp32 → ~2 GB of 16 GB used). Estimated p95 latency
    # ~150–300 ms at seq_len=2048 — comfortably inside the 500 ms target.
    # Same Turing (sm_75) architecture as the training cluster, so the trained
    # checkpoint runs through PyTorch SDPA attention unchanged.
    instance_type="Standard_NC16as_T4_v3",
    instance_count=1,
    environment_variables={
        "MIN_CONFIDENCE": "0.70",      # Below 70% → UNKNOWN → human review queue
        "UNKNOWN_LABEL":  "UNKNOWN",
        # MAX_SEQ_LEN is consumed by score.py to cap tokenisation length at
        # serve time. Tunable here without redeploying the model: drop to 1024
        # or 512 if a latency regression appears under load (Phase H harness).
        "MAX_SEQ_LEN":    "2048",
    },
)
ml_client.begin_create_or_update(deployment).wait()

# Route all traffic to this deployment
ml_client.online_endpoints.begin_create_or_update(
    ManagedOnlineEndpoint(
        name=endpoint_name,
        traffic={"default": 100},
    )
).wait()
print(f"✓ Endpoint '{endpoint_name}' deployed on Standard_NC16as_T4_v3 (1× T4)")
print(f"  Confidence threshold: 70%  |  MAX_SEQ_LEN: 2048")
print(f"  Predictions below 70% confidence will have requires_human_review=true")


### Test the endpoint

In [ ]:
import json
import tempfile

# Test 1: Email with explicit IMO — expect high confidence, no human review
test_explicit = {
    "emailSubject": "RFQ - urgent",
    "emailBody":    "Please quote for vessel IMO 9319466.",
    "Attachments":  "RFQ.pdf: Vessel Ever Given; IMO 9319466",
    "emailAddresses": "buyer@example.com supplier@example.com",
}

# Test 2: Email with only implicit signals — expect lower confidence, may need review
test_implicit = {
    "emailSubject":   "Parts order ref 22b",
    "emailBody":      "Please send the parts as per our standing order.",
    "Attachments":    "RFQ234-22b.pdf",
    "emailAddresses": "supplier@example.com",
}

def invoke_endpoint(payload: dict) -> dict:
    tmp = tempfile.NamedTemporaryFile("w", suffix=".json", delete=False)
    json.dump(payload, tmp)
    tmp.close()
    raw = ml_client.online_endpoints.invoke(
        endpoint_name=endpoint_name,
        deployment_name="default",
        request_file=tmp.name,
    )
    return json.loads(raw)

print("── Test 1: explicit IMO in email ──")
r1 = invoke_endpoint(test_explicit)
print(json.dumps(r1, indent=2))
# Expected: predicted_imo="IMO9319466", confidence>0.70, requires_human_review=false

print("\n── Test 2: implicit signal only ──")
r2 = invoke_endpoint(test_implicit)
print(json.dumps(r2, indent=2))
# Expected: predicted_imo=<vessel IMO or UNKNOWN>, requires_human_review=true if conf<0.70
